# MAS. Activity_1

**Authors:** 
* Adriana Sandoval Santos - A01667445
* Marbella Barba De los Santos - A01660946
* Javier Padilla Marcial - A01660945
* Camila Quintanilla Cabrera - A01662397
* Alani Daniel Garcia Maya - A01277308 

**Date:** November 23rd, 2026

**GitHub repository:** https://github.com/Adrisandoval2006/MAS.-Activity-2.git

## 1. Objective

The objective of this activity is compare Max-Plus coordinated cleaning robots with random-movement robots in a 20 × 30 grid containing 100 initially dirty cells. Evaluate the effect of 1, 2, 4, 8, and 16 robots over a maximum of 500 steps, using 100 repetitions per configuration. Report success probability, cleaning time, and movements. Results are calculated from experiments; no outcomes are assumed in advance.

## 2. Agents, Model and Implementation

* **Agent.py** defines coordinated and random robots. 
* **Coordination.py** builds a Manhattan-distance coordination graph and computes actions using Max-Plus. 
* **Model.py** initializes the grid, dirty cells, robots and simulation steps. 
* **Test.py** runs the experiments and saves CSV files. 
* **View.py** visualizes a single run.

Methodological note: the random agent moves in eight directions and cleans before moving, while the coordinated agent uses orthogonal actions and checks for cleaning after moving. This difference should be acknowledged when interpreting comparisons.

## 3. Experimental Setup

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from Test import TESTS

experiments = pd.DataFrame(TESTS)
display(experiments[["method", "numAgents", "width", "height", "nDirty", "maxSteps", "repetitions"]])
print("Expected simulations:", experiments["repetitions"].sum())

## 4. Data Collection

Run the experiments once. This may take some time, especially for 16-agent Max-Plus configurations. If the CSVs already exist, skip this cell and load them below.

In [ ]:
from Test import runAll, saveCsv, OUTPUT_FILE, AGENT_OUTPUT_FILE, COLUMNS, AGENT_COLUMNS

summaries, agentRows = runAll(TESTS)
saveCsv(summaries, OUTPUT_FILE, COLUMNS)
saveCsv(agentRows, AGENT_OUTPUT_FILE, AGENT_COLUMNS)

In [ ]:
data = pd.read_csv("test_results.csv")
agentData = pd.read_csv("agent_results.csv")
data["method"] = data["test"].str.extract(r"^(max plus|random)", expand=False).replace({"max plus": "maxplus"})
agentData["method"] = agentData["test"].str.extract(r"^(max plus|random)", expand=False).replace({"max plus": "maxplus"})
print("Simulation records:", len(data))
print("Robot records:", len(agentData))
display(data.head())
display(agentData.head())

## 5. Data validation

Check that each experiment contains the expected number of runs, the total robot movements match the recorded total, and the completion flags are consistent.

In [ ]:
counts = data.groupby(["method", "numAgents"]).size().unstack(0)
display(counts)
assert len(data) == 1000, "Expected 1,000 simulation records"
assert (counts == 100).all().all(), "Expected 100 runs per configuration"
assert (data["executionTime"] <= data["maxSteps"]).all()
assert (data["allClean"] == (data["remainingDirtyCells"] == 0)).all()
assert data["nDirty"].eq(100).all()
individualTotals = (agentData.groupby(["test", "run"])["movements"]
                    .sum().reset_index(name="individualTotal"))
validation = data.merge(individualTotals, on=["test", "run"], validate="one_to_one")
assert len(validation) == len(data)
assert (validation["totalMoves"] == validation["individualTotal"]).all()
print("All validations passed!")

## 6. Metrics and results

Success probability is the fraction of runs where all dirty cells were cleaned. Average completion time is calculated only from successful runs; unsuccessful runs are excluded rather than treated as completed at step 500. Average execution steps includes all runs. Movements per cleaned cell uses the actual number of dirty cells cleaned.

In [ ]:
data["cellsCleaned"] = data["nDirty"] - data["remainingDirtyCells"]
data["movementsPerCleanedCell"] = data["totalMoves"] / data["cellsCleaned"].replace(0, float("nan"))
results = (data.groupby(["method", "numAgents"])
    .agg(runs=("run", "count"),
         successProbability=("allClean", "mean"),
         averageExecutionSteps=("executionTime", "mean"),
         averageCompletionSteps=("completionTime", "mean"),
         averageMoves=("totalMoves", "mean"),
         averageCleanPercentage=("cleanedPercentage", "mean"),
         averageMovementsPerCell=("movementsPerCleanedCell", "mean"),
         averageRunningTimeS=("runTimeS", "mean"))
    .reset_index())
results["successRatePercent"] = results["successProbability"] * 100
display(results.round(2))

## 7. Graph 1 — Success probability by method and number of robots

In [ ]:
pivot = results.pivot(index="numAgents", columns="method", values="successRatePercent")
ax = pivot.plot(kind="bar", figsize=(9, 5))
ax.set(xlabel="Number of agents", ylabel="Successful runs (%)", title="Probability of cleaning all cells within 500 steps")
ax.set_ylim(0, 105)
ax.legend(title="Method")
plt.tight_layout()
plt.show()

## 8. Graph 2 — Average total movements

In [ ]:
ax = results.pivot(index="numAgents", columns="method", values="averageMoves").plot(marker="o", figsize=(9, 5))
ax.set(xlabel="Number of agents", ylabel="Average total movements", title="Movement comparison: Max-Plus vs Random")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 9. Graph 3 — Mean completion time (successful runs only)

In [ ]:
ax = results.pivot(index="numAgents", columns="method", values="averageCompletionSteps").plot(marker="o", figsize=(9, 5))
ax.set(xlabel="Number of agents", ylabel="Mean steps to completion", title="Cleaning time for successful simulations")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()
print("Missing values mean there were no successful runs in that configuration.")

## 10. Graph 4 — Average percentage of initially dirty cells cleaned

In [ ]:
ax = results.pivot(index="numAgents", columns="method", values="averageCleanPercentage").plot(marker="o", figsize=(9, 5))
ax.set(xlabel="Number of agents", ylabel="Initially dirty cells cleaned (%)", title="Cleaning coverage")
ax.set_ylim(0, 105)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 11. Graph 5 — Average movements per cleaned cell

In [ ]:
ax = results.pivot(index="numAgents", columns="method", values="averageMovementsPerCell").plot(marker="o", figsize=(9, 5))
ax.set(xlabel="Number of agents", ylabel="Movements per cleaned cell", title="Movement efficiency")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 12. Graph 6 — Average program running time

In [ ]:
ax = results.pivot(index="numAgents", columns="method", values="averageRunningTimeS").plot(marker="o", figsize=(9, 5))
ax.set(xlabel="Number of agents", ylabel="Average wall-clock time (seconds)", title="Computational cost")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 13. Discussion

Complete after executing the experiments. Discuss: (1) whether Max-Plus improves success probability, (2) how the number of robots affects cleaning time and movements, (3) whether computational time increases with coordination, and (4) the limitations of comparing algorithms with different movement/cleaning rules. Avoid interpreting missing completion-time values as zero.

## 14. Conclusions
Complete after inspecting the actual tables and plots. State which configurations performed best under each metric and distinguish cleaning effectiveness from movement efficiency and computation cost.

## 15. Sample simulation snapshot

Run python View.py separately, pause the visualization, and save a screenshot for the report. The notebook plots summarize the experimental results rather than showing a spatial snapshot.